# 02.1 - Customer Fact Table

Goal: build a customer-grain fact table with lifecycle, order, value, payment, shipping, return, and review metrics for downstream analysis and modeling.

## Setup

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from datathon_2026_r1.eda import load_all_tables

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:.4f}".format)

PLOT_RCPARAMS = {
    "figure.figsize": (11, 6),
    "figure.dpi": 120,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.grid": True,
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.8,
    "legend.frameon": False,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "lines.linewidth": 2.0,
    "lines.markersize": 5,
    "patch.edgecolor": "white",
    "patch.linewidth": 0.5,
}
plt.rcParams.update(PLOT_RCPARAMS)
sns.set_theme(style="whitegrid", palette="Set2", rc=PLOT_RCPARAMS)

DERIVED_DATA_DIR = PROJECT_ROOT / "data" / "derived"
DERIVED_DATA_DIR.mkdir(parents=True, exist_ok=True)

## Load Raw Tables

In [2]:
tables = load_all_tables()
customers = tables["customers"].copy()
products = tables["products"].copy()
orders = tables["orders"].copy()
order_items = tables["order_items"].copy()
payments = tables["payments"].copy()
shipments = tables["shipments"].copy()
returns = tables["returns"].copy()
reviews = tables["reviews"].copy()
inventory = tables["inventory"].copy()

{
    "customers": customers.shape,
    "products": products.shape,
    "orders": orders.shape,
    "order_items": order_items.shape,
    "payments": payments.shape,
    "shipments": shipments.shape,
    "returns": returns.shape,
    "reviews": reviews.shape,
    "inventory": inventory.shape,
}

{'customers': (121930, 7),
 'products': (2412, 8),
 'orders': (646945, 8),
 'order_items': (714669, 7),
 'payments': (646945, 4),
 'shipments': (566067, 4),
 'returns': (39939, 7),
 'reviews': (113551, 7),
 'inventory': (60247, 17)}

## Helper Functions

In [3]:
def most_frequent(series: pd.Series):
    non_null = series.dropna()
    if non_null.empty:
        return pd.NA
    counts = non_null.value_counts()
    return counts.index.sort_values()[0] if len(counts) > 1 and counts.iloc[0] == counts.iloc[1] else counts.idxmax()


def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator.div(denominator.where(denominator.ne(0)))

## Build Order-level Enrichment

In [4]:
order_items_enriched = order_items.merge(
    products[["product_id", "cogs"]],
    on="product_id",
    how="left",
)
order_items_enriched["gross_item_value"] = order_items_enriched["quantity"] * order_items_enriched["unit_price"]
order_items_enriched["discount_amount"] = order_items_enriched["discount_amount"].fillna(0)
order_items_enriched["net_item_value"] = order_items_enriched["gross_item_value"] - order_items_enriched["discount_amount"]
order_items_enriched["item_cogs"] = order_items_enriched["quantity"] * order_items_enriched["cogs"]

order_item_fact = (
    order_items_enriched.groupby("order_id", as_index=False)
    .agg(
        distinct_products=("product_id", "nunique"),
        total_quantity=("quantity", "sum"),
        gross_revenue=("gross_item_value", "sum"),
        total_discount_amount=("discount_amount", "sum"),
        net_revenue=("net_item_value", "sum"),
        total_cogs=("item_cogs", "sum"),
    )
)
order_item_fact["gross_margin"] = order_item_fact["net_revenue"] - order_item_fact["total_cogs"]

order_level_fact = (
    orders.merge(order_item_fact, on="order_id", how="left")
    .merge(payments, on="order_id", how="left", suffixes=("", "_payment"))
    .merge(shipments, on="order_id", how="left")
)
order_level_fact = order_level_fact.sort_values(["customer_id", "order_date", "order_id"]).reset_index(drop=True)
order_level_fact["payment_method_recorded"] = order_level_fact["payment_method_payment"].combine_first(
    order_level_fact["payment_method"]
)
order_level_fact.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,distinct_products,total_quantity,gross_revenue,total_discount_amount,net_revenue,total_cogs,gross_margin,payment_method_payment,payment_value,installments,ship_date,delivery_date,shipping_fee,payment_method_recorded
0,5280,2012-07-25,1,15201,delivered,cod,desktop,paid_search,1,1,12627.3900,0.0000,12627.3900,10040.8610,2586.5290,cod,12627.3900,1,2012-07-28,2012-08-02,0.3800,cod
1,184922,2014-05-31,1,15201,returned,credit_card,mobile,referral,1,1,1478.7800,0.0000,1478.7800,1383.1158,95.6642,credit_card,1478.7800,3,2014-05-31,2014-06-03,28.9100,credit_card
2,308113,2015-07-31,1,15201,delivered,cod,mobile,paid_search,1,8,44708.3200,0.0000,44708.3200,41380.4790,3327.8410,cod,44708.3200,1,2015-08-02,2015-08-06,1.3200,cod
3,483190,2017-04-23,1,15201,delivered,cod,mobile,paid_search,1,7,37213.4000,0.0000,37213.4000,20974.5951,16238.8049,cod,37213.4000,1,2017-04-24,2017-04-30,2.0900,cod
4,702081,2020-02-24,1,15201,delivered,credit_card,mobile,organic_search,2,6,4233.3300,0.0000,4233.3300,3113.2216,1120.1084,credit_card,4233.3300,1,2020-02-25,2020-02-28,27.8400,credit_card


## Build Order Fact Table

In [5]:
order_return_fact = (
    returns.groupby("order_id", as_index=False)
    .agg(
        return_records_count=("return_id", "nunique"),
        returned_products_count=("product_id", "nunique"),
        returned_units_count=("return_quantity", "sum"),
        total_refund_amount=("refund_amount", "sum"),
        first_return_date=("return_date", "min"),
        last_return_date=("return_date", "max"),
    )
)

order_review_fact = (
    reviews.groupby("order_id", as_index=False)
    .agg(
        reviews_count=("review_id", "nunique"),
        reviewed_products_count=("product_id", "nunique"),
        avg_review_rating=("rating", "mean"),
        first_review_date=("review_date", "min"),
        latest_review_date=("review_date", "max"),
    )
)

order_fact_table = (
    order_level_fact.merge(order_return_fact, on="order_id", how="left")
    .merge(order_review_fact, on="order_id", how="left")
)

order_fact_table["order_month"] = order_fact_table["order_date"].dt.to_period("M").dt.to_timestamp()
order_fact_table["ship_lead_time_days"] = (
    order_fact_table["ship_date"] - order_fact_table["order_date"]
).dt.days
order_fact_table["delivery_lead_time_days"] = (
    order_fact_table["delivery_date"] - order_fact_table["order_date"]
).dt.days
order_fact_table["days_to_first_return"] = (
    order_fact_table["first_return_date"] - order_fact_table["order_date"]
).dt.days
order_fact_table["days_to_first_review"] = (
    order_fact_table["first_review_date"] - order_fact_table["order_date"]
).dt.days
order_fact_table["total_order_value"] = order_fact_table["gross_revenue"]
order_fact_table["net_revenue_proxy"] = order_fact_table["payment_value"]
order_fact_table["payment_gap_vs_net_revenue"] = (
    order_fact_table["net_revenue_proxy"] - order_fact_table["net_revenue"]
)
order_fact_table["discount_rate"] = safe_divide(
    order_fact_table["total_discount_amount"],
    order_fact_table["gross_revenue"],
)
order_fact_table["gross_margin_rate"] = safe_divide(
    order_fact_table["gross_margin"],
    order_fact_table["net_revenue"],
)
order_fact_table["refund_to_revenue_ratio"] = safe_divide(
    order_fact_table["total_refund_amount"],
    order_fact_table["net_revenue"],
)

count_columns = [
    "return_records_count",
    "returned_products_count",
    "returned_units_count",
    "reviews_count",
    "reviewed_products_count",
]
order_fact_table[count_columns] = order_fact_table[count_columns].fillna(0).astype("Int64")

order_fact_columns = [
    "order_id",
    "order_date",
    "order_month",
    "customer_id",
    "zip",
    "order_status",
    "payment_method_recorded",
    "device_type",
    "order_source",
    "distinct_products",
    "total_quantity",
    "total_order_value",
    "total_discount_amount",
    "net_revenue",
    "net_revenue_proxy",
    "payment_gap_vs_net_revenue",
    "total_cogs",
    "gross_margin",
    "discount_rate",
    "gross_margin_rate",
    "payment_value",
    "installments",
    "ship_date",
    "delivery_date",
    "shipping_fee",
    "ship_lead_time_days",
    "delivery_lead_time_days",
    "return_records_count",
    "returned_products_count",
    "returned_units_count",
    "total_refund_amount",
    "first_return_date",
    "last_return_date",
    "days_to_first_return",
    "refund_to_revenue_ratio",
    "reviews_count",
    "reviewed_products_count",
    "avg_review_rating",
    "first_review_date",
    "latest_review_date",
    "days_to_first_review",
]
order_fact_table = order_fact_table[order_fact_columns]

order_fact_table.head()

,order_id,order_date,order_month,customer_id,zip,order_status,payment_method_recorded,device_type,order_source,distinct_products,total_quantity,total_order_value,total_discount_amount,net_revenue,net_revenue_proxy,payment_gap_vs_net_revenue,total_cogs,gross_margin,discount_rate,gross_margin_rate,payment_value,installments,ship_date,delivery_date,shipping_fee,ship_lead_time_days,delivery_lead_time_days,return_records_count,returned_products_count,returned_units_count,total_refund_amount,first_return_date,last_return_date,days_to_first_return,refund_to_revenue_ratio,reviews_count,reviewed_products_count,avg_review_rating,first_review_date,latest_review_date,days_to_first_review
0,5280,2012-07-25,2012-07-01,1,15201,delivered,cod,desktop,paid_search,1,1,12627.3900,0.0000,12627.3900,12627.3900,0.0000,10040.8610,2586.5290,0.0000,0.2048,12627.3900,1,2012-07-28,2012-08-02,0.3800,3.0000,8.0000,0,0,0,NaN,NaT,NaT,NaN,NaN,0,0,NaN,NaT,NaT,NaN
1,184922,2014-05-31,2014-05-01,1,15201,returned,credit_card,mobile,referral,1,1,1478.7800,0.0000,1478.7800,1478.7800,0.0000,1383.1158,95.6642,0.0000,0.0647,1478.7800,3,2014-05-31,2014-06-03,28.9100,0.0000,3.0000,1,1,1,1398.6400,2014-06-08,2014-06-08,8.0000,0.9458,0,0,NaN,NaT,NaT,NaN
2,308113,2015-07-31,2015-07-01,1,15201,delivered,cod,mobile,paid_search,1,8,44708.3200,0.0000,44708.3200,44708.3200,0.0000,41380.4790,3327.8410,0.0000,0.0744,44708.3200,1,2015-08-02,2015-08-06,1.3200,2.0000,6.0000,0,0,0,NaN,NaT,NaT,NaN,NaN,0,0,NaN,NaT,NaT,NaN
3,483190,2017-04-23,2017-04-01,1,15201,delivered,cod,mobile,paid_search,1,7,37213.4000,0.0000,37213.4000,37213.4000,0.0000,20974.5951,16238.8049,0.0000,0.4364,37213.4000,1,2017-04-24,2017-04-30,2.0900,1.0000,7.0000,0,0,0,NaN,NaT,NaT,NaN,NaN,0,0,NaN,NaT,NaT,NaN
4,702081,2020-02-24,2020-02-01,1,15201,delivered,credit_card,mobile,organic_search,2,6,4233.3300,0.0000,4233.3300,4233.3300,0.0000,3113.2216,1120.1084,0.0000,0.2646,4233.3300,1,2020-02-25,2020-02-28,27.8400,1.0000,4.0000,0,0,0,NaN,NaT,NaT,NaN,NaN,0,0,NaN,NaT,NaT,NaN


## Build Customer Fact Table

In [6]:
customer_order_sequence = order_level_fact[["customer_id", "order_id", "order_date"]].copy()
customer_order_sequence["order_rank"] = customer_order_sequence.groupby("customer_id").cumcount() + 1
customer_order_dates = (
    customer_order_sequence.pivot(index="customer_id", columns="order_rank", values="order_date")
    .reindex(columns=[1, 2])
    .rename(columns={1: "first_order_date", 2: "second_order_date"})
    .reset_index()
)

customer_item_fact = (
    order_items[["order_id", "product_id", "quantity"]]
    .merge(orders[["order_id", "customer_id"]], on="order_id", how="left")
    .groupby("customer_id", as_index=False)
    .agg(
        distinct_products_count=("product_id", "nunique"),
        total_quantity=("quantity", "sum"),
    )
)

customer_order_fact = (
    order_level_fact.groupby("customer_id", as_index=False)
    .agg(
        last_order_date=("order_date", "max"),
        orders_count=("order_id", "nunique"),
        delivered_orders_count=("order_status", lambda s: int(s.eq("delivered").sum())),
        returned_orders_count=("order_status", lambda s: int(s.eq("returned").sum())),
        cancelled_orders_count=("order_status", lambda s: int(s.eq("cancelled").sum())),
        shipped_orders_count=("ship_date", lambda s: int(s.notna().sum())),
        gross_revenue=("gross_revenue", "sum"),
        total_discount_amount=("total_discount_amount", "sum"),
        net_revenue=("net_revenue", "sum"),
        total_cogs=("total_cogs", "sum"),
        gross_margin=("gross_margin", "sum"),
        total_payment_value=("payment_value", "sum"),
        avg_payment_installments=("installments", "mean"),
        total_shipping_fee=("shipping_fee", "sum"),
        avg_order_value=("net_revenue", "mean"),
        avg_items_per_order=("total_quantity", "mean"),
        preferred_payment_method=("payment_method_recorded", most_frequent),
        preferred_device_type=("device_type", most_frequent),
        preferred_order_source=("order_source", most_frequent),
    )
)

customer_return_fact = (
    returns.groupby("order_id", as_index=False)
    .agg(
        return_records_per_order=("return_id", "nunique"),
        returned_units_per_order=("return_quantity", "sum"),
        refund_amount_per_order=("refund_amount", "sum"),
        latest_return_date=("return_date", "max"),
    )
    .merge(orders[["order_id", "customer_id"]], on="order_id", how="left")
    .groupby("customer_id", as_index=False)
    .agg(
        return_records_count=("return_records_per_order", "sum"),
        returned_units_count=("returned_units_per_order", "sum"),
        total_refund_amount=("refund_amount_per_order", "sum"),
        last_return_date=("latest_return_date", "max"),
    )
)

customer_review_fact = (
    reviews.groupby("customer_id", as_index=False)
    .agg(
        reviews_count=("review_id", "nunique"),
        avg_review_rating=("rating", "mean"),
        latest_review_date=("review_date", "max"),
    )
)

analysis_end_date = orders["order_date"].max()

customer_fact_table = (
    customers.merge(customer_order_dates, on="customer_id", how="left")
    .merge(customer_order_fact, on="customer_id", how="left")
    .merge(customer_item_fact, on="customer_id", how="left")
    .merge(customer_return_fact, on="customer_id", how="left")
    .merge(customer_review_fact, on="customer_id", how="left")
)

customer_fact_table["customer_tenure_days"] = (
    customer_fact_table["last_order_date"] - customer_fact_table["first_order_date"]
).dt.days
customer_fact_table["days_since_last_order"] = (
    analysis_end_date - customer_fact_table["last_order_date"]
).dt.days
customer_fact_table["days_from_signup_to_first_order"] = (
    customer_fact_table["first_order_date"] - customer_fact_table["signup_date"]
).dt.days
customer_fact_table["days_to_second_order"] = (
    customer_fact_table["second_order_date"] - customer_fact_table["first_order_date"]
).dt.days
customer_fact_table["total_order_value"] = customer_fact_table["gross_revenue"]
customer_fact_table["net_revenue_proxy"] = customer_fact_table["total_payment_value"]
customer_fact_table["payment_gap_vs_net_revenue"] = (
    customer_fact_table["net_revenue_proxy"] - customer_fact_table["net_revenue"]
)
customer_fact_table["reviews_count"] = customer_fact_table["reviews_count"].fillna(0)
customer_fact_table["return_rate_orders"] = safe_divide(
    customer_fact_table["returned_orders_count"],
    customer_fact_table["orders_count"],
)
customer_fact_table["review_rate_orders"] = safe_divide(
    customer_fact_table["reviews_count"],
    customer_fact_table["orders_count"],
)
customer_fact_table["refund_to_revenue_ratio"] = safe_divide(
    customer_fact_table["total_refund_amount"],
    customer_fact_table["net_revenue"],
)

integer_like_columns = [
    "orders_count",
    "delivered_orders_count",
    "returned_orders_count",
    "cancelled_orders_count",
    "shipped_orders_count",
    "distinct_products_count",
    "total_quantity",
    "return_records_count",
    "returned_units_count",
    "reviews_count",
    "customer_tenure_days",
    "days_since_last_order",
    "days_from_signup_to_first_order",
    "days_to_second_order",
]
for column in integer_like_columns:
    if column in customer_fact_table:
        customer_fact_table[column] = customer_fact_table[column].astype("Int64")

customer_fact_columns = [
    "customer_id",
    "zip",
    "city",
    "signup_date",
    "gender",
    "age_group",
    "acquisition_channel",
    "first_order_date",
    "second_order_date",
    "last_order_date",
    "customer_tenure_days",
    "days_since_last_order",
    "days_from_signup_to_first_order",
    "days_to_second_order",
    "orders_count",
    "delivered_orders_count",
    "returned_orders_count",
    "cancelled_orders_count",
    "shipped_orders_count",
    "distinct_products_count",
    "total_quantity",
    "total_order_value",
    "total_discount_amount",
    "net_revenue",
    "net_revenue_proxy",
    "payment_gap_vs_net_revenue",
    "total_cogs",
    "gross_margin",
    "total_payment_value",
    "avg_payment_installments",
    "total_shipping_fee",
    "avg_order_value",
    "avg_items_per_order",
    "preferred_payment_method",
    "preferred_device_type",
    "preferred_order_source",
    "return_records_count",
    "returned_units_count",
    "total_refund_amount",
    "last_return_date",
    "return_rate_orders",
    "refund_to_revenue_ratio",
    "reviews_count",
    "avg_review_rating",
    "latest_review_date",
    "review_rate_orders",
]
customer_fact_table = customer_fact_table[customer_fact_columns]

customer_fact_table.head()

,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel,first_order_date,second_order_date,last_order_date,customer_tenure_days,days_since_last_order,days_from_signup_to_first_order,days_to_second_order,orders_count,delivered_orders_count,returned_orders_count,cancelled_orders_count,shipped_orders_count,distinct_products_count,total_quantity,total_order_value,total_discount_amount,net_revenue,net_revenue_proxy,payment_gap_vs_net_revenue,total_cogs,gross_margin,total_payment_value,avg_payment_installments,total_shipping_fee,avg_order_value,avg_items_per_order,preferred_payment_method,preferred_device_type,preferred_order_source,return_records_count,returned_units_count,total_refund_amount,last_return_date,return_rate_orders,refund_to_revenue_ratio,reviews_count,avg_review_rating,latest_review_date,review_rate_orders
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,2012-07-25,2014-05-31,2021-04-24,3195,616,-3445,675,6,4,1,0,5,7,26,142803.4700,0.0000,142803.4700,142803.4700,0.0000,107825.5787,34977.8913,142803.4700,2.1667,60.5400,23800.5783,4.3333,cod,mobile,paid_search,1,1,1398.6400,2014-06-08,0.1667,0.0098,0,NaN,NaT,0.0000
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign,2013-09-20,2013-10-28,2022-07-06,3211,178,-98,38,4,2,0,2,2,4,23,225225.9400,20532.0500,204693.8900,204693.8900,0.0000,222763.1731,-18069.2831,204693.8900,6.7500,4.7100,51173.4725,5.7500,apple_pay,desktop,direct,<NA>,<NA>,NaN,NaT,0.0000,NaN,2,2.0000,2022-07-22,0.5000
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search,2012-08-27,2012-11-24,2013-07-29,336,3442,-2157,89,3,2,0,1,2,3,12,52093.4700,0.0000,52093.4700,52093.4700,0.0000,35187.4515,16906.0185,52093.4700,1.0000,27.3700,17364.4900,4.0000,apple_pay,mobile,social_media,<NA>,<NA>,NaN,NaT,0.0000,NaN,1,5.0000,2012-09-14,0.3333
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral,2020-06-28,NaT,2020-06-28,0,916,942,<NA>,1,1,0,0,1,1,8,13340.3200,2401.2600,10939.0600,10939.0600,0.0000,15450.2675,-4511.2075,10939.0600,1.0000,2.3900,10939.0600,8.0000,credit_card,mobile,social_media,<NA>,<NA>,NaN,NaT,0.0000,NaN,0,NaN,NaT,0.0000
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search,2012-08-09,2012-11-26,2019-03-27,2421,1375,-3697,109,5,5,0,0,5,5,23,69075.3400,4895.4800,64179.8600,64179.8600,0.0000,51443.3114,12736.5486,64179.8600,2.4000,35.5800,12835.9720,4.6000,credit_card,desktop,social_media,<NA>,<NA>,NaN,NaT,0.0000,NaN,1,5.0000,2015-08-06,0.2000


## Build Cohort Fact Tables

In [7]:
customer_cohort_base = order_fact_table[[
    "customer_id",
    "order_id",
    "order_month",
    "total_order_value",
    "net_revenue",
    "net_revenue_proxy",
    "gross_margin",
]].merge(
    customer_fact_table[[
        "customer_id",
        "signup_date",
        "first_order_date",
        "acquisition_channel",
    ]],
    on="customer_id",
    how="left",
)
customer_cohort_base["first_order_cohort_month"] = (
    customer_cohort_base["first_order_date"].dt.to_period("M").dt.to_timestamp()
)
customer_cohort_base["signup_cohort_month"] = (
    customer_cohort_base["signup_date"].dt.to_period("M").dt.to_timestamp()
)
customer_cohort_base["cohort_index"] = (
    (customer_cohort_base["order_month"].dt.year - customer_cohort_base["first_order_cohort_month"].dt.year) * 12
    + (customer_cohort_base["order_month"].dt.month - customer_cohort_base["first_order_cohort_month"].dt.month)
)

customer_cohort_activity_fact = (
    customer_cohort_base.groupby(
        [
            "customer_id",
            "first_order_cohort_month",
            "signup_cohort_month",
            "order_month",
            "cohort_index",
            "acquisition_channel",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        orders_count=("order_id", "nunique"),
        total_order_value=("total_order_value", "sum"),
        net_revenue=("net_revenue", "sum"),
        net_revenue_proxy=("net_revenue_proxy", "sum"),
        gross_margin=("gross_margin", "sum"),
    )
    .sort_values(["customer_id", "order_month"])
    .reset_index(drop=True)
)
customer_cohort_activity_fact["active_month_flag"] = 1
customer_cohort_activity_fact["payment_gap_vs_net_revenue"] = (
    customer_cohort_activity_fact["net_revenue_proxy"] - customer_cohort_activity_fact["net_revenue"]
)

customer_cohort_activity_fact.head()

,customer_id,first_order_cohort_month,signup_cohort_month,order_month,cohort_index,acquisition_channel,orders_count,total_order_value,net_revenue,net_revenue_proxy,gross_margin,active_month_flag,payment_gap_vs_net_revenue
0,1,2012-07-01,2021-12-01,2012-07-01,0,social_media,1,12627.3900,12627.3900,12627.3900,2586.5290,1,0.0000
1,1,2012-07-01,2021-12-01,2014-05-01,22,social_media,1,1478.7800,1478.7800,1478.7800,95.6642,1,0.0000
2,1,2012-07-01,2021-12-01,2015-07-01,36,social_media,1,44708.3200,44708.3200,44708.3200,3327.8410,1,0.0000
3,1,2012-07-01,2021-12-01,2017-04-01,57,social_media,1,37213.4000,37213.4000,37213.4000,16238.8049,1,0.0000
4,1,2012-07-01,2021-12-01,2020-02-01,91,social_media,1,4233.3300,4233.3300,4233.3300,1120.1084,1,0.0000


In [8]:
signup_activation_fact = customer_fact_table[[
    "customer_id",
    "signup_date",
    "first_order_date",
    "second_order_date",
    "acquisition_channel",
    "orders_count",
    "days_from_signup_to_first_order",
    "net_revenue",
]].copy()
signup_activation_fact["signup_cohort_month"] = signup_activation_fact["signup_date"].dt.to_period("M").dt.to_timestamp()
signup_activation_fact["activation_month"] = signup_activation_fact["first_order_date"].dt.to_period("M").dt.to_timestamp()
signup_activation_fact["months_to_activation"] = (
    (signup_activation_fact["activation_month"].dt.year - signup_activation_fact["signup_cohort_month"].dt.year) * 12
    + (signup_activation_fact["activation_month"].dt.month - signup_activation_fact["signup_cohort_month"].dt.month)
)
signup_activation_fact.loc[signup_activation_fact["first_order_date"].isna(), "months_to_activation"] = pd.NA
signup_activation_fact["activated_flag"] = signup_activation_fact["first_order_date"].notna().astype(int)
signup_activation_fact["repeat_order_flag"] = signup_activation_fact["second_order_date"].notna().astype(int)
signup_activation_fact["orders_count"] = signup_activation_fact["orders_count"].fillna(0).astype("Int64")
signup_activation_fact["num_orders"] = signup_activation_fact["orders_count"]
signup_activation_fact["days_signup_to_first_order"] = signup_activation_fact["days_from_signup_to_first_order"]
signup_activation_fact["signup_before_first_order_flag"] = (
    signup_activation_fact["days_signup_to_first_order"].ge(0).fillna(False).astype(int)
)

signup_activation_columns = [
    "customer_id",
    "signup_date",
    "signup_cohort_month",
    "first_order_date",
    "second_order_date",
    "activation_month",
    "months_to_activation",
    "days_signup_to_first_order",
    "signup_before_first_order_flag",
    "activated_flag",
    "repeat_order_flag",
    "num_orders",
    "acquisition_channel",
    "net_revenue",
]
signup_activation_fact = signup_activation_fact[signup_activation_columns]

signup_activation_fact.head()

,customer_id,signup_date,signup_cohort_month,first_order_date,second_order_date,activation_month,months_to_activation,days_signup_to_first_order,signup_before_first_order_flag,activated_flag,repeat_order_flag,num_orders,acquisition_channel,net_revenue
0,1,2021-12-30,2021-12-01,2012-07-25,2014-05-31,2012-07-01,-113.0000,-3445,0,1,1,6,social_media,142803.4700
1,2,2013-12-27,2013-12-01,2013-09-20,2013-10-28,2013-09-01,-3.0000,-98,0,1,1,4,email_campaign,204693.8900
2,3,2018-07-24,2018-07-01,2012-08-27,2012-11-24,2012-08-01,-71.0000,-2157,0,1,1,3,organic_search,52093.4700
3,4,2017-11-29,2017-11-01,2020-06-28,NaT,2020-06-01,31.0000,942,1,1,0,1,referral,10939.0600
4,5,2022-09-23,2022-09-01,2012-08-09,2012-11-26,2012-08-01,-121.0000,-3697,0,1,1,5,organic_search,64179.8600


## Build Product-month Inventory Fact Table

In [9]:
product_month_inventory_fact = inventory.copy()
product_month_inventory_fact["snapshot_month"] = (
    product_month_inventory_fact["snapshot_date"].dt.to_period("M").dt.to_timestamp()
)
product_month_inventory_fact["stockout_rate_days"] = safe_divide(
    product_month_inventory_fact["stockout_days"],
    product_month_inventory_fact["snapshot_date"].dt.days_in_month,
)
product_month_inventory_fact["available_units"] = (
    product_month_inventory_fact["stock_on_hand"] + product_month_inventory_fact["units_sold"]
)
product_month_inventory_fact["net_units_change"] = (
    product_month_inventory_fact["units_received"] - product_month_inventory_fact["units_sold"]
)

product_month_inventory_columns = [
    "product_id",
    "snapshot_date",
    "snapshot_month",
    "year",
    "month",
    "product_name",
    "category",
    "segment",
    "stock_on_hand",
    "units_received",
    "units_sold",
    "net_units_change",
    "available_units",
    "stockout_days",
    "stockout_rate_days",
    "days_of_supply",
    "fill_rate",
    "sell_through_rate",
    "stockout_flag",
    "overstock_flag",
    "reorder_flag",
]
product_month_inventory_fact = product_month_inventory_fact[product_month_inventory_columns]

product_month_inventory_fact.head()

,product_id,snapshot_date,snapshot_month,year,month,product_name,category,segment,stock_on_hand,units_received,units_sold,net_units_change,available_units,stockout_days,stockout_rate_days,days_of_supply,fill_rate,sell_through_rate,stockout_flag,overstock_flag,reorder_flag
0,1,2022-10-31,2022-10-01,2022,10,DragonWear MA-01,Casual,All-weather,3,1,1,0,4,2,0.0645,90.0000,0.9333,0.2500,1,0,0
1,1,2022-11-30,2022-11-01,2022,11,DragonWear MA-01,Casual,All-weather,3,1,1,0,4,1,0.0333,90.0000,0.9667,0.2500,1,0,0
2,1,2022-12-31,2022-12-01,2022,12,DragonWear MA-01,Casual,All-weather,3,1,1,0,4,1,0.0323,90.0000,0.9667,0.2500,1,0,0
3,3,2016-04-30,2016-04-01,2016,4,DragonWear MA-03,Casual,All-weather,35,13,11,2,46,2,0.0667,95.5000,0.9333,0.2391,1,1,0
4,3,2016-05-31,2016-05-01,2016,5,DragonWear MA-03,Casual,All-weather,36,11,10,1,46,1,0.0323,108.0000,0.9667,0.2174,1,1,0


## Fact Table Quality Checks

In [10]:
pd.DataFrame(
    [
        {
            "table": "order_fact_table",
            "rows": len(order_fact_table),
            "unique_key": int(order_fact_table["order_id"].nunique()),
            "duplicate_key_rows": int(order_fact_table.duplicated(["order_id"]).sum()),
            "records_with_activity": int(order_fact_table["order_id"].notna().sum()),
            "records_without_activity": 0,
        },
        {
            "table": "customer_fact_table",
            "rows": len(customer_fact_table),
            "unique_key": int(customer_fact_table["customer_id"].nunique()),
            "duplicate_key_rows": int(customer_fact_table.duplicated(["customer_id"]).sum()),
            "records_with_activity": int(customer_fact_table["orders_count"].fillna(0).gt(0).sum()),
            "records_without_activity": int(customer_fact_table["orders_count"].fillna(0).eq(0).sum()),
        },
        {
            "table": "customer_cohort_activity_fact",
            "rows": len(customer_cohort_activity_fact),
            "unique_key": int(
                customer_cohort_activity_fact[["customer_id", "order_month"]]
                .drop_duplicates()
                .shape[0]
            ),
            "duplicate_key_rows": int(
                customer_cohort_activity_fact.duplicated(["customer_id", "order_month"]).sum()
            ),
            "records_with_activity": int(customer_cohort_activity_fact["active_month_flag"].eq(1).sum()),
            "records_without_activity": 0,
        },
        {
            "table": "signup_activation_fact",
            "rows": len(signup_activation_fact),
            "unique_key": int(signup_activation_fact["customer_id"].nunique()),
            "duplicate_key_rows": int(signup_activation_fact.duplicated(["customer_id"]).sum()),
            "records_with_activity": int(signup_activation_fact["activated_flag"].eq(1).sum()),
            "records_without_activity": int(signup_activation_fact["activated_flag"].eq(0).sum()),
        },
        {
            "table": "product_month_inventory_fact",
            "rows": len(product_month_inventory_fact),
            "unique_key": int(
                product_month_inventory_fact[["product_id", "snapshot_month"]]
                .drop_duplicates()
                .shape[0]
            ),
            "duplicate_key_rows": int(
                product_month_inventory_fact.duplicated(["product_id", "snapshot_month"]).sum()
            ),
            "records_with_activity": int(product_month_inventory_fact["product_id"].notna().sum()),
            "records_without_activity": 0,
        },
    ]
)

,table,rows,unique_key,duplicate_key_rows,records_with_activity,records_without_activity
0,order_fact_table,646945,646945,0,646945,0
1,customer_fact_table,121930,121930,0,90246,31684
2,customer_cohort_activity_fact,586976,586976,0,586976,0
3,signup_activation_fact,121930,121930,0,90246,31684
4,product_month_inventory_fact,60247,60247,0,60247,0


In [11]:
customer_fact_table[[
    "customer_id",
    "signup_date",
    "acquisition_channel",
    "first_order_date",
    "second_order_date",
    "last_order_date",
    "orders_count",
    "total_order_value",
    "net_revenue",
    "net_revenue_proxy",
    "gross_margin",
    "total_payment_value",
    "return_records_count",
    "reviews_count",
    "avg_review_rating",
    "preferred_order_source",
]].head(10)

,customer_id,signup_date,acquisition_channel,first_order_date,second_order_date,last_order_date,orders_count,total_order_value,net_revenue,net_revenue_proxy,gross_margin,total_payment_value,return_records_count,reviews_count,avg_review_rating,preferred_order_source
0,1,2021-12-30,social_media,2012-07-25,2014-05-31,2021-04-24,6,142803.4700,142803.4700,142803.4700,34977.8913,142803.4700,1,0,NaN,paid_search
1,2,2013-12-27,email_campaign,2013-09-20,2013-10-28,2022-07-06,4,225225.9400,204693.8900,204693.8900,-18069.2831,204693.8900,<NA>,2,2.0000,direct
2,3,2018-07-24,organic_search,2012-08-27,2012-11-24,2013-07-29,3,52093.4700,52093.4700,52093.4700,16906.0185,52093.4700,<NA>,1,5.0000,social_media
3,4,2017-11-29,referral,2020-06-28,NaT,2020-06-28,1,13340.3200,10939.0600,10939.0600,-4511.2075,10939.0600,<NA>,0,NaN,social_media
4,5,2022-09-23,organic_search,2012-08-09,2012-11-26,2019-03-27,5,69075.3400,64179.8600,64179.8600,12736.5486,64179.8600,<NA>,1,5.0000,social_media
5,6,2022-04-14,organic_search,2012-10-28,2015-01-01,2021-06-20,10,154268.2100,148527.2600,148527.2600,18147.2056,148527.2600,<NA>,1,5.0000,email_campaign
6,8,2015-09-11,social_media,2012-10-27,2013-05-09,2017-01-01,6,165106.4500,158356.4100,158356.4100,35486.5368,158356.4100,<NA>,1,4.0000,email_campaign
7,9,2020-02-14,email_campaign,2019-06-05,NaT,2019-06-05,1,15926.5200,15926.5200,15926.5200,4100.2868,15926.5200,<NA>,0,NaN,organic_search
8,10,2014-03-03,organic_search,2012-08-30,2013-04-11,2022-05-06,12,330933.2300,321108.5500,321108.5500,57651.6449,321108.5500,1,1,4.0000,organic_search
9,11,2017-11-07,organic_search,2014-08-02,2016-08-03,2017-05-15,4,221384.7600,221384.7600,221384.7600,67964.5483,221384.7600,1,1,5.0000,social_media


## Export Fact Table

In [12]:
order_fact_table_path = DERIVED_DATA_DIR / "order_fact_table.csv"
customer_fact_table_path = DERIVED_DATA_DIR / "customer_fact_table.csv"
customer_cohort_activity_fact_path = DERIVED_DATA_DIR / "customer_cohort_activity_fact.csv"
signup_activation_fact_path = DERIVED_DATA_DIR / "signup_activation_fact.csv"
product_month_inventory_fact_path = DERIVED_DATA_DIR / "product_month_inventory_fact.csv"

order_fact_table.to_csv(order_fact_table_path, index=False)
customer_fact_table.to_csv(customer_fact_table_path, index=False)
customer_cohort_activity_fact.to_csv(customer_cohort_activity_fact_path, index=False)
signup_activation_fact.to_csv(signup_activation_fact_path, index=False)
product_month_inventory_fact.to_csv(product_month_inventory_fact_path, index=False)

{
    "order_fact_table": order_fact_table_path,
    "customer_fact_table": customer_fact_table_path,
    "customer_cohort_activity_fact": customer_cohort_activity_fact_path,
    "signup_activation_fact": signup_activation_fact_path,
    "product_month_inventory_fact": product_month_inventory_fact_path,
}

{'order_fact_table': WindowsPath('D:/NOHOPE/Python/Datathon_VinUni_2026/DATATHON-2026-r1/data/derived/order_fact_table.csv'),
 'customer_fact_table': WindowsPath('D:/NOHOPE/Python/Datathon_VinUni_2026/DATATHON-2026-r1/data/derived/customer_fact_table.csv'),
 'customer_cohort_activity_fact': WindowsPath('D:/NOHOPE/Python/Datathon_VinUni_2026/DATATHON-2026-r1/data/derived/customer_cohort_activity_fact.csv'),
 'signup_activation_fact': WindowsPath('D:/NOHOPE/Python/Datathon_VinUni_2026/DATATHON-2026-r1/data/derived/signup_activation_fact.csv'),
 'product_month_inventory_fact': WindowsPath('D:/NOHOPE/Python/Datathon_VinUni_2026/DATATHON-2026-r1/data/derived/product_month_inventory_fact.csv')}